In [2]:
from pyspark.sql import SparkSession
from pyspark.errors import AnalysisException
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [3]:
spark = SparkSession.builder.appName("Analysis of Customers and Transactions.").remote("sc://localhost:15002").getOrCreate()

# setting error level as we don't want to see too many unimportant errors - this no longer works as errors are at the
# server level when using ssh
# spark.sparkContext.setLogLevel("ERROR")

# Step 1 - read in all raw data from CSVs and show result
customers_df_path = "data/customers"
transactions_df_path = "data/transactions"

customers_df = (spark.read.option("header",True).option("ignoreLeadingWhiteSpace", True)
                .option("ignoreTrailingWhiteSpace", True).csv(customers_df_path))
transactions_df = (spark.read.option("header",True).option("ignoreLeadingWhiteSpace", True)
                   .option("ignoreTrailingWhiteSpace", True).csv(transactions_df_path))

In [4]:
print("\n=== customers schema ===")
customers_df.printSchema()

print("\n=== transactions schema ===")
transactions_df.printSchema()

# Stage 2 - get into tempview and apply sql to clean data
## table summaries - shows counts of rows with mean and other output values.

print("\n=== customers summary ===")
customers_df.summary().show()

print("\n=== transactions summary ===")
transactions_df.summary().show()



=== customers schema ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: string (nullable = true)


=== transactions schema ===
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- amount: string (nullable = true)


=== customers summary ===
+-------+---------------+-------------+--------------------+-------+------+-----------------+
|summary|    customer_id|customer_name|               email|   city| state|registration_date|
+-------+---------------+-------------+--------------------+-------+------+-----------------+
|  count|         606000|       606000|              601578| 603129|603129|           606000|
|   mean|           NULL|    

In [5]:
## dropDuplicates - safe to run now as it is looking for completely identical whole rows
customers = customers_df.dropDuplicates()
transactions = transactions_df.dropDuplicates()

print("\n=== customers summary post duplicate removal ===")
customers_df.summary().show()

print("\n=== transactions summary post duplicate removal ===")
transactions_df.summary().show()

## create temp views for dataframes
customers_df.createOrReplaceTempView("customers")
transactions_df.createOrReplaceTempView("transactions")

## truncate and show table names
### truncate true can applies a 20 char max limit and false ignores any limits and you can also enter a fixed number.
### True also right aligns and False left aligns the resultset
### vertical changes the view of the output to vertical and you can also set n number of outputs.
### Instead of "SHOW TABLES" other options such as "DESCRIBE TABLE" also exist
spark.sql("SHOW TABLES").show(truncate=True, vertical=True)


=== customers summary post duplicate removal ===
+-------+---------------+-------------+--------------------+-------+------+-----------------+
|summary|    customer_id|customer_name|               email|   city| state|registration_date|
+-------+---------------+-------------+--------------------+-------+------+-----------------+
|  count|         606000|       606000|              601578| 603129|603129|           606000|
|   mean|           NULL|         NULL|                NULL|   NULL|  NULL|             NULL|
| stddev|           NULL|         NULL|                NULL|   NULL|  NULL|             NULL|
|    min|  cust-0000007 | Andrew Allen|andrew.allen10011...|Atlanta|    AZ|       2018-01-01|
|    25%|           NULL|         NULL|                NULL|   NULL|  NULL|             NULL|
|    50%|           NULL|         NULL|                NULL|   NULL|  NULL|             NULL|
|    75%|           NULL|         NULL|                NULL|   NULL|  NULL|             NULL|
|    max| 

In [6]:
## Clean customers temp view

spark.sql("""
    CREATE OR REPLACE TEMP VIEW customers_clean AS
    SELECT customer_id,
           customer_name,
           city,
           state,
           registration_date
    FROM (
        SELECT nullif(trim(customer_id), '') AS customer_id,

               nullif(initcap(regexp_replace(trim(customer_name), '\\s+', ' ')), '')
                   AS customer_name,

               nullif(lower(regexp_replace(trim(city), '\\s+', ' ')), '')
                   AS city,

               nullif(lower(regexp_replace(trim(state), '\\s+', ' ')), '')
                   AS state,

               cast(registration_date AS date) AS registration_date
        FROM customers
    )
    WHERE customer_id IS NOT NULL
    QUALIFY row_number() OVER (
        PARTITION BY customer_id
        ORDER BY registration_date DESC
    ) = 1
""")
## show/confirm creation of this new temp view.
spark.sql("SHOW TABLES").show(truncate=True, vertical=True)

print("\n=== customers_clean summary after cleaning ===")

## run the sql below and you can see it has removed all nulls from name and registration date and all duplicates removed
spark.sql("""
    SELECT count(*)                                  AS total_rows,

           count(customer_id)                        AS customer_id_non_null,
           count(DISTINCT customer_id)               AS customer_id_distinct,

           count(customer_name)                      AS name_non_null,
           count(*) - count(customer_name)           AS name_nulls,

           count(city)                               AS city_non_null,
           count(DISTINCT city)                      AS city_distinct,
           count(*) - count(city)                    AS city_nulls,

           count(state)                              AS state_non_null,
           count(DISTINCT state)                     AS state_distinct,
           count(*) - count(state)                   AS state_nulls,

           min(registration_date)                    AS earliest_registration,
           max(registration_date)                    AS latest_registration,
           count(*) - count(registration_date)       AS registration_date_nulls
    FROM customers_clean
""").show(vertical=True)

-RECORD 0----------------------
 namespace   |                 
 tableName   | customers       
 isTemporary | true            
-RECORD 1----------------------
 namespace   |                 
 tableName   | customers_clean 
 isTemporary | true            
-RECORD 2----------------------
 namespace   |                 
 tableName   | transactions    
 isTemporary | true            


=== customers_clean summary after cleaning ===
-RECORD 0-----------------------------
 total_rows              | 600000     
 customer_id_non_null    | 600000     
 customer_id_distinct    | 600000     
 name_non_null           | 600000     
 name_nulls              | 0          
 city_non_null           | 597157     
 city_distinct           | 30         
 city_nulls              | 2843       
 state_non_null          | 597157     
 state_distinct          | 19         
 state_nulls             | 2843       
 earliest_registration   | 2018-01-01 
 latest_registration     | 2025-12-10 
 registration_date_nu

In [7]:
# for the above we can alternatively see summary output after converting to spark dataframe, but it is not as good
spark.table("customers_clean").summary().show(truncate=False)

## show random 10 entries
spark.sql("""
    SELECT *
    FROM customers_clean
    ORDER BY rand()
    LIMIT 10
""").show(truncate=False)

## Analysis of customers
## states with counts

spark.sql("""
    SELECT state, count(*) 
    FROM customers_clean
    GROUP BY state
    ORDER BY count(*) 
""").show(truncate=False)

## earliest registration date by state
spark.sql("""
    SELECT customer_id, customer_name, state, registration_date,
           row_number() OVER (
               PARTITION BY state
               ORDER BY registration_date ASC NULLS LAST, customer_id
           ) AS signup_order
    FROM customers_clean
    QUALIFY signup_order = 1
    ORDER BY state
""").show(truncate=False)

# latest registration date by state
spark.sql("""
    SELECT customer_id, customer_name, state, registration_date,
           row_number() OVER (
               PARTITION BY state
               ORDER BY registration_date DESC, customer_id
           ) AS signup_order
    FROM customers_clean
    QUALIFY signup_order = 1
    ORDER BY state
""").show(51, truncate=False)

## This time for dropping duplicates in transactions we will use the spark sql instead of dropDuplicates

# Cleaning of transactions
spark.sql("""
    SELECT *
    FROM transactions
    LIMIT 10
""").show(truncate=True)

+-------+------------+-------------+-------+------+
|summary|customer_id |customer_name|city   |state |
+-------+------------+-------------+-------+------+
|count  |600000      |600000       |597157 |597157|
|mean   |NULL        |NULL         |NULL   |NULL  |
|stddev |NULL        |NULL         |NULL   |NULL  |
|min    |CUST-0000001|A Hley Allen |atlanta|az    |
|25%    |NULL        |NULL         |NULL   |NULL  |
|50%    |NULL        |NULL         |NULL   |NULL  |
|75%    |NULL        |NULL         |NULL   |NULL  |
|max    |cust-0599957|William Young|tampa  |wa    |
+-------+------------+-------------+-------+------+

+------------+------------------+------------+-----+-----------------+
|customer_id |customer_name     |city        |state|registration_date|
+------------+------------------+------------+-----+-----------------+
|CUST-0182094|Elizabeth Thomp On|new york    |ny   |2022-08-14       |
|CUST-0020003|Daniel Ander On   |columbu     |oh   |2023-10-21       |
|CUST-0086977|Michel

In [ ]:
## drop duplicates using distinct sql statement instead of dropDuplicates on dataframe
spark.sql("SELECT DISTINCT * FROM transactions").createOrReplaceTempView("transactions_duplicates_removed")
#confirm new temp view creation
spark.sql("SHOW TABLES").show(truncate=True, vertical=True)

# below summaries show the difference after dropping complete identical rows
spark.table("transactions_duplicates_removed").summary().show(truncate=False)
spark.table("transactions").summary().show(truncate=False)

# this is just to show how to remove duplicates based on a Single column
(spark.table("transactions_duplicates_removed")
 .dropDuplicates(["product"]).createOrReplaceTempView("transactions_dedup_single_col"))

(spark.table("transactions_duplicates_removed")
 .dropDuplicates(["product", "customer_id"])
 .createOrReplaceTempView("transactions_dedup_multi_col"))

# keeping the latest by transaction date with window function

spark.sql("""
    CREATE OR REPLACE TEMP VIEW transactions_latest AS
    SELECT *
    FROM transactions_duplicates_removed
    QUALIFY row_number() OVER (
        PARTITION BY product, customer_id
        ORDER BY transaction_date DESC, transaction_id DESC
    ) = 1
""")

spark.sql("""
    SELECT customer_id, count(*) AS entries
    FROM transactions_duplicates_removed
    GROUP BY customer_id
    HAVING count(*) < 10
    ORDER BY entries DESC
""").show(100, truncate=False)

spark.sql("""
    SELECT *
    FROM transactions_dedup_single_col
    ORDER BY product
    LIMIT 100
""").show(truncate=True)

# now to clean the transactions table like customers was done

spark.sql("""
    SELECT *
    FROM transactions_duplicates_removed
    LIMIT 10
""").show(truncate=True)

-RECORD 0---------------------------
 namespace   |                      
 tableName   | customers            
 isTemporary | true                 
-RECORD 1---------------------------
 namespace   |                      
 tableName   | customers_clean      
 isTemporary | true                 
-RECORD 2---------------------------
 namespace   |                      
 tableName   | transactions         
 isTemporary | true                 
-RECORD 3---------------------------
 namespace   |                      
 tableName   | transactions_dupl... 
 isTemporary | true                 

+-------+--------------+---------------+---------+----------------+------------------+------------------+
|summary|transaction_id|customer_id    |product  |transaction_date|quantity          |amount            |
+-------+--------------+---------------+---------+----------------+------------------+------------------+
|count  |2900000       |2891237        |2900000  |2900000         |2896044           |289

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW transactions_clean AS
    SELECT transaction_id,
           customer_id,
           product,
           transaction_date,
           quantity,
           amount
    FROM (
        SELECT nullif(upper(trim(transaction_id)), '')          AS transaction_id,

               nullif(trim(customer_id), '')                    AS customer_id,

               nullif(lower(regexp_replace(trim(product), '\\s+', ' ')), '')
                                                                AS product,

               try_cast(trim(transaction_date) AS date)         AS transaction_date,

               try_cast(trim(quantity) AS int)                  AS quantity,

               try_cast(trim(amount) AS decimal(12, 2))         AS amount
        FROM transactions_duplicates_removed
    )
    WHERE transaction_id IS NOT NULL
    QUALIFY row_number() OVER (
        PARTITION BY transaction_id
        ORDER BY transaction_date DESC
    ) = 1
""")

# now show result

spark.sql("""
    SELECT count(*)                                  AS total_rows,

           count(transaction_id)                     AS transaction_id_non_null,
           count(DISTINCT transaction_id)            AS transaction_id_distinct,

           count(customer_id)                        AS customer_id_non_null,
           count(DISTINCT customer_id)               AS customer_id_distinct,
           count(*) - count(customer_id)             AS customer_id_nulls,

           count(product)                            AS product_non_null,
           count(DISTINCT product)                   AS product_distinct,
           count(*) - count(product)                 AS product_nulls,

           min(transaction_date)                     AS earliest_transaction,
           max(transaction_date)                     AS latest_transaction,
           count(*) - count(transaction_date)        AS transaction_date_nulls,

           min(quantity)                             AS quantity_min,
           max(quantity)                             AS quantity_max,
           sum(quantity)                             AS quantity_total,
           count(*) - count(quantity)                AS quantity_nulls,
           count_if(quantity <= 0)                   AS quantity_zero_or_negative,

           min(amount)                               AS amount_min,
           max(amount)                               AS amount_max,
           round(avg(amount), 2)                     AS amount_avg,
           sum(amount)                               AS amount_total,
           count(*) - count(amount)                  AS amount_nulls,
           count_if(amount < 0)                      AS amount_negative
    FROM transactions_clean
""").show(vertical=True)

In [ ]:
# now time to join customers with transactions
# Inner join
spark.sql("""
    SELECT t.transaction_id,
           c.customer_id,
           c.customer_name,
           t.amount
    FROM transactions_clean AS t
    INNER JOIN customers_clean AS c
        ON t.customer_id = c.customer_id
    """).show(truncate=False)